# Scikit-learn model adapters for `mnplib`

This notebook demonstrates the modular scikit-learn adapter package for the simplified `mnplib` classes.

The core metric classes remain model-agnostic:

- `Miscoding` works with a selected feature subset.
- `Inaccuracy` works with predictions.
- `Surfeit` works with a model-description string.
- `Nescience` combines the three explicit artifacts.

The adapter layer converts supported scikit-learn models into those artifacts:

```python
model + X_eval -> subset, predictions, model_string
```

Supported model families are fixed by the library:

- decision trees
- linear regression
- logistic regression
- linear SVM/SVR
- Naive Bayes
- MLP neural networks

Unsupported estimators fail explicitly; there is no dynamic registry or generic fallback serialization.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression
from sklearn.dummy import DummyRegressor
from sklearn.exceptions import NotFittedError
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from mnplib.nescience import Nescience
from mnplib.models import (
    SerializationConfig,
    components_model,
    explain_model,
    nescience_model,
    score_model,
    sklearn_model_artifacts,
)

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Supported models and canonical schema

The adapter package uses a fixed internal list of serializer classes. Each serializer extracts:

1. the feature subset used by the model;
2. the predictions on an evaluation dataset;
3. a canonical string representation of the fitted model.

All model strings use the same schema:

```text
SCHEMA canonical_nescience_model_v1
MODEL <EstimatorClass>
TASK <classification|regression>
INPUTS <selected feature names>
PARAMETERS
    ...
RULE
    ...
```


In [ ]:
SUPPORTED_ADAPTER_MODEL_TYPES = (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    LinearRegression,
    LogisticRegression,
    LinearSVC,
    LinearSVR,
    GaussianNB,
    MLPClassifier,
    MLPRegressor,
)

print("Fixed adapter model types:")
for model_type in SUPPORTED_ADAPTER_MODEL_TYPES:
    print("-", model_type.__name__)


## 3. Helper functions for tables and plots

In [ ]:
def regression_row(name, model, metric, X_eval, y_eval, *, feature_names=None, config=None):
    """Return one comparison row for a regression model."""
    artifacts = sklearn_model_artifacts(
        model,
        X_eval,
        feature_names=feature_names,
        config=config,
    )
    components = metric.components(**artifacts.to_nescience_kwargs())

    return {
        "model": name,
        "r2": r2_score(y_eval, artifacts.predictions),
        "rmse": mean_squared_error(y_eval, artifacts.predictions) ** 0.5,
        "nescience": metric.aggregate_components(**components),
        **components,
        "n_features_in_use": len(artifacts.subset),
        "description_length": len(artifacts.model_string.encode("utf-8")),
        "model_type": artifacts.model_type,
    }


def classification_row(name, model, metric, X_eval, y_eval, *, feature_names=None, config=None):
    """Return one comparison row for a classification model."""
    artifacts = sklearn_model_artifacts(
        model,
        X_eval,
        feature_names=feature_names,
        config=config,
    )
    components = metric.components(**artifacts.to_nescience_kwargs())

    return {
        "model": name,
        "accuracy": accuracy_score(y_eval, artifacts.predictions),
        "nescience": metric.aggregate_components(**components),
        **components,
        "n_features_in_use": len(artifacts.subset),
        "description_length": len(artifacts.model_string.encode("utf-8")),
        "model_type": artifacts.model_type,
    }


def plot_components(df, title):
    """Plot the four nescience components."""
    component_cols = ["deficiency", "surplus", "inaccuracy", "surfeit"]
    df.set_index("model")[component_cols].plot(kind="bar", figsize=(12, 5))
    plt.ylabel("Component value")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()


def plot_nescience(df, title):
    """Plot scalar nescience values."""
    df.set_index("model")["nescience"].plot(kind="bar", figsize=(10, 4))
    plt.ylabel("Nescience")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()


def explanation_row(name, model, metric, X_eval, *, feature_names=None, config=None):
    """Return one compact explanation row."""
    explanation = explain_model(
        metric,
        model,
        X_eval,
        feature_names=feature_names,
        config=config,
    )
    return {
        "model": name,
        "nescience": explanation["nescience"],
        "dominant_component": explanation["dominant_component"],
        "profile": explanation["profile"],
        "model_type": explanation["model_type"],
        "n_features_in_use": explanation["model_metadata"]["n_features_in_use"],
    }


## 4. Regression example: tree versus linear models

This example compares supported regression estimators using the same canonical serialization schema:

- `DecisionTreeRegressor`
- `LinearRegression`
- `LinearSVR`
- `MLPRegressor`

The models are trained on the training split. Nescience is evaluated on the test split, so the internal `Nescience` object is fitted on `(X_test, y_test)`.


In [ ]:
X, y = make_regression(
    n_samples=160,
    n_features=8,
    n_informative=4,
    noise=10.0,
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=42,
)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]

regression_models = {
    "tree_depth_3": DecisionTreeRegressor(max_depth=3, random_state=42),
    "linear": LinearRegression(),
    "linear_svr": LinearSVR(max_iter=10000, random_state=42),
    "mlp": MLPRegressor(hidden_layer_sizes=(8,), max_iter=200, random_state=42),
}

for model in regression_models.values():
    model.fit(X_train, y_train)

config = SerializationConfig(precision=4, include_metadata=True)

regression_metric = Nescience(
    X_type="numeric",
    y_type="numeric",
    aggregation="euclidean",
    n_bins=4,
).fit(X_test, y_test)

regression_results = pd.DataFrame(
    [
        regression_row(
            name,
            model,
            regression_metric,
            X_test,
            y_test,
            feature_names=feature_names,
            config=config,
        )
        for name, model in regression_models.items()
    ]
).sort_values("nescience")

regression_results


In [ ]:
plot_components(regression_results, "Regression models: four nescience components")
plot_nescience(regression_results, "Regression models: scalar nescience")


### 4.1 Inspecting canonical model strings

The tree and the linear model are different model families, but both are serialized into the same canonical schema.


In [ ]:
tree_artifacts = sklearn_model_artifacts(
    regression_models["tree_depth_3"],
    X_test,
    feature_names=feature_names,
    config=config,
)

linear_artifacts = sklearn_model_artifacts(
    regression_models["linear"],
    X_test,
    feature_names=feature_names,
    config=config,
)

print("DecisionTreeRegressor canonical string:")
print(tree_artifacts.model_string[:1800])

print("\nLinearRegression canonical string:")
print(linear_artifacts.model_string[:1800])


### 4.2 Direct artifact workflow versus convenience wrapper

The adapter can be used explicitly through `sklearn_model_artifacts()`, or directly through `nescience_model()`.


In [ ]:
model = regression_models["linear_svr"]

artifacts = sklearn_model_artifacts(
    model,
    X_test,
    feature_names=feature_names,
    config=config,
)

print(artifacts.model_string[:1200])


## 5. Classification example: decision tree versus logistic regression

In [ ]:
Xc, yc = make_classification(
    n_samples=180,
    n_features=6,
    n_informative=3,
    n_redundant=0,
    random_state=42,
)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc,
    yc,
    test_size=0.35,
    random_state=42,
)
classification_feature_names = [f"feature_{i}" for i in range(Xc.shape[1])]

classification_models = {
    "tree_depth_3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "tree_unrestricted": DecisionTreeClassifier(random_state=42),
    "logistic": LogisticRegression(max_iter=1000),
    "linear_svc": LinearSVC(max_iter=10000, random_state=42),
    "gaussian_nb": GaussianNB(),
    "mlp": MLPClassifier(hidden_layer_sizes=(8,), max_iter=200, random_state=42),
}

for model in classification_models.values():
    model.fit(Xc_train, yc_train)

classification_metric = Nescience(
    X_type="numeric",
    y_type="categorical",
    aggregation="euclidean",
    n_bins=4,
).fit(Xc_test, yc_test)

classification_results = pd.DataFrame(
    [
        classification_row(
            name,
            model,
            classification_metric,
            Xc_test,
            yc_test,
            feature_names=classification_feature_names,
            config=config,
        )
        for name, model in classification_models.items()
    ]
).sort_values("nescience")

classification_results


In [ ]:
plot_components(classification_results, "Classification models: four nescience components")
plot_nescience(classification_results, "Classification models: scalar nescience")


## 6. Diagnostic explanations

`explain_model()` adds model metadata to the simplified `Nescience.explain(...)` output.


In [ ]:
regression_explanations = pd.DataFrame(
    [
        explanation_row(
            name,
            model,
            regression_metric,
            X_test,
            feature_names=feature_names,
            config=config,
        )
        for name, model in regression_models.items()
    ]
).sort_values("nescience")

regression_explanations


In [ ]:
classification_explanations = pd.DataFrame(
    [
        explanation_row(
            name,
            model,
            classification_metric,
            Xc_test,
            feature_names=classification_feature_names,
            config=config,
        )
        for name, model in classification_models.items()
    ]
).sort_values("nescience")

classification_explanations


## 7. Changing the practical objective with weights

The canonical adapter only extracts artifacts. The practical objective remains controlled by `Nescience`.

Here we compare three weight settings:

- balanced;
- accuracy-focused;
- simplicity-focused.


In [ ]:
weight_settings = {
    "balanced": None,
    "accuracy_focused": {
        "deficiency": 1.0,
        "surplus": 1.0,
        "inaccuracy": 3.0,
        "surfeit": 1.0,
    },
    "simplicity_focused": {
        "deficiency": 1.0,
        "surplus": 2.0,
        "inaccuracy": 1.0,
        "surfeit": 3.0,
    },
}

weighted_rows = []

for setting_name, weights in weight_settings.items():
    weighted_metric = Nescience(
        X_type="numeric",
        y_type="numeric",
        aggregation="euclidean",
        weights=weights,
        n_bins="auto",
    ).fit(X_test, y_test)

    for model_name, model in regression_models.items():
        value = nescience_model(
            weighted_metric,
            model,
            X_test,
            feature_names=feature_names,
            config=config,
        )
        weighted_rows.append(
            {
                "setting": setting_name,
                "model": model_name,
                "nescience": value,
            }
        )

weighted_results = pd.DataFrame(weighted_rows)

weighted_results.pivot(
    index="model",
    columns="setting",
    values="nescience",
)


In [ ]:
weighted_results.pivot(
    index="model",
    columns="setting",
    values="nescience",
).plot(kind="bar", figsize=(12, 5))

plt.ylabel("Weighted nescience")
plt.title("Effect of component weights")
plt.xticks(rotation=45)
plt.show()


## 8. Strict handling of unsupported models

Unsupported models should fail explicitly instead of silently assuming that all features are used. This is important because a wrong subset would corrupt the miscoding component.


In [ ]:
dummy = DummyRegressor().fit(X_train, y_train)

try:
    sklearn_model_artifacts(dummy, X_test)
except ValueError as exc:
    print(type(exc).__name__)
    print(exc)


## 9. Summary

The adapter layer makes the simplified `mnplib` design practical without compromising its separation of concerns.

The core metrics remain simple and explicit:

```python
metric.nescience(subset=..., predictions=..., model_string=...)
```

The scikit-learn adapter adds a practical bridge:

```python
sklearn_model_artifacts(model, X_eval)
nescience_model(metric, model, X_eval)
components_model(metric, model, X_eval)
explain_model(metric, model, X_eval)
score_model(metric, model, X_eval)
```

The most important design choice is the canonical serialization schema. It makes surfeit comparisons more meaningful than relying on heterogeneous model-specific string representations.
